In [3]:
%%file generator.py
import json, os, random, time
from datetime import datetime

output_dir = "data/stream"
os.makedirs(output_dir, exist_ok=True)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

print("Uruchamiam generator...")
while True:
    batch = [generate_transaction() for _ in range(2)]
    filename = f"{output_dir}/events_{int(time.time())}.json"
    with open(filename, "w") as f:
        for e in batch:
            f.write(json.dumps(e) + "\n")
    print(f"Wrote: {filename}")
    time.sleep(5)

Overwriting generator.py


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import count, avg, round as _round

In [2]:
spark = SparkSession.builder.appName("Lab3_PracaDomowa_Agregacja").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [4]:
tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

In [5]:
stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))

In [6]:
agregacja_sklepy = (
    stream.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_transakcji"),
        _round(avg("amount"), 2).alias("srednia_kwota_PLN")
    )
)

In [7]:
def tabela(df, batch_id, tstop=5):
    print(f"--- Tabela (Batch) nr: {batch_id} ---")
    df.show(truncate=False)
    if batch_id == tstop:
        print("Koniec obliczeń!")
        df.sparkSession.streams.active[0].stop()

print("Obliczam statystyki na żywo... (Czekam na dane z generatora)")
query = (agregacja_sklepy.writeStream
         .outputMode("complete")
         .foreachBatch(tabela)
         .start())

Obliczam statystyki na żywo... (Czekam na dane z generatora)
--- Tabela (Batch) nr: 0 ---
+--------+-----------------+-----------------+
|store   |liczba_transakcji|srednia_kwota_PLN|
+--------+-----------------+-----------------+
|Gdańsk  |66               |2460.81          |
|Kraków  |76               |2531.29          |
|Warszawa|65               |2601.46          |
|Wrocław |73               |2372.14          |
+--------+-----------------+-----------------+

--- Tabela (Batch) nr: 1 ---
+--------+-----------------+-----------------+
|store   |liczba_transakcji|srednia_kwota_PLN|
+--------+-----------------+-----------------+
|Gdańsk  |69               |2450.56          |
|Kraków  |77               |2549.34          |
|Warszawa|67               |2585.3           |
|Wrocław |75               |2377.61          |
+--------+-----------------+-----------------+

--- Tabela (Batch) nr: 2 ---
+--------+-----------------+-----------------+
|store   |liczba_transakcji|srednia_kwota_PLN|
+---